In [57]:
# 1. Pulling Multi-Year Stock Data
print("1. Pulling Multi-Year Stock Data\n")
import yfinance as yf
import pandas as pd
import datetime

# Pulling SPY, APPL, MSFT, from 2019 - 2024
tickers = ['SPY','AAPL','MSFT']
data = yf.download(tickers, start='2019-01-01', end='2024-12-31', auto_adjust=True) # see notes on auto adjust in a_u3

data = data['Close']

# Inspecting the DatetimeIndex
print(data.index)
print(type(data.index)) # Should be DatatimeIndex

print(type(data))

print("\nLooking at .dt accessor")
#data.loc['2023'].dt.year -> This won't work! only works on Series (not dataframes)
    #.dt accessor is for Series with datetime values, not for accessing the index
# Get years from the index
years = data.index.year
# Or for filtered data
years_2023 = data.loc['2023'].index.year
print(years_2023)  # All values will be 2023

[*********************100%***********************]  3 of 3 completed

1. Pulling Multi-Year Stock Data

DatetimeIndex(['2019-01-02', '2019-01-03', '2019-01-04', '2019-01-07',
               '2019-01-08', '2019-01-09', '2019-01-10', '2019-01-11',
               '2019-01-14', '2019-01-15',
               ...
               '2024-12-16', '2024-12-17', '2024-12-18', '2024-12-19',
               '2024-12-20', '2024-12-23', '2024-12-24', '2024-12-26',
               '2024-12-27', '2024-12-30'],
              dtype='datetime64[s]', name='Date', length=1509, freq=None)
<class 'pandas.DatetimeIndex'>
<class 'pandas.DataFrame'>

Looking at .dt accessor
Index([2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023,
       ...
       2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023],
      dtype='int32', name='Date', length=250)


In [40]:
print("2. Partial String Indexing Practice\n")
# Selecting all of Q3 2023 data (July - September)
q3_2023 = data.loc['2023-07':'2023-09']
# print(q3_2023)

#converting to period index with quarterly frequency
quarterly = data.to_period('Q')
# print(quarterly)
q3_2023_alt = quarterly.loc['2023Q3'].to_timestamp()
# print(q3_2023_alt)

# Selecting all Fridays in 2923
# Friday = 4 (Monday=0, Tuesday=1,...,Friday=4,...)
fridays_2023 = data[(data.index.year == 2023)&(data.index.dayofweek == 4)]
#print(fridays_2023)

# last trading day per month in 2023
last_days = data.loc['2023'].resample('ME').last()
        # QE also works. MS is month start
print(last_days)

2. Partial String Indexing Practice

Ticker            AAPL        MSFT         SPY
Date                                          
2023-01-31  142.012665  241.472321  389.670929
2023-02-28  145.304947  243.649887  379.873596
2023-03-31  162.545197  281.630432  393.958466
2023-04-30  167.256912  300.151794  400.251892
2023-05-31  174.960495  321.494293  402.099579
2023-06-30  191.464539  333.389160  428.155518
2023-07-31  193.912476  328.866150  442.170471
2023-08-31  185.693756  321.556854  434.984253
2023-09-30  169.226730  309.774139  414.351044
2023-10-31  168.791824  331.710938  405.356079
2023-11-30  187.996994  372.493378  442.382843
2023-12-31  190.550461  369.671967  462.579956


For future reference:

| **Old** | **New** | **Meaning** |
|---------|---------|-------------|
| `'M'`   | `'ME'`  | Month End   |
| `'Q'`   | `'QE'`  | Quarter End |
| `'Y'`   | `'YE'`  | Year End    |
| `'MS'`  | `'MS'`  | Month Start (unchanged) |
| `'D'`   | `'D'`   | Day (unchanged) |
| `'B'`   | `'B'`   | Business Day (unchanged) |

---

Good catch! Does it work with `'ME'` now? 🚀

In [49]:
# # 3. Timezone Conversion
# # yfinance data is timezone-naive (i.e there is not timezone info attached) - we need to localise to UTC
# if data.index.tz is None:
#     # Naive → make it aware (assign timezone)
#     data.index = data.index.tz_localize('UTC')
# else:
#     # Already aware → convert to UTC (in case it's something else)
#     data.index = data.index.tz_convert('UTC')
#     #using checking function

# #converting to US/Eastern (NYSE timezone)
# data.index = data.index.tz_convert('US/Eastern')

# # Checking the result
# print(data.index[0]) # Should show timezone information

# 4. Business Day Calendar Validation
from pandas.tseries.offsets import CustomBusinessDay
from pandas.tseries.holiday import USFederalHolidayCalendar

# Generate NYSE trading dats for 2023
nyse_cal = CustomBusinessDay(calendar=USFederalHolidayCalendar())
trading_days_2023 = pd.date_range('2023-01-01','2023-12-31', freq=nyse_cal)

# TIMEZONE-SAFE COMPARISON: Use string dates
actual_dates = set(data.loc['2023'].index.strftime('%Y-%m-%d'))
expected_dates = set(trading_days_2023.strftime('%Y-%m-%d'))

# Finding mismatches
missing = expected_dates - actual_dates
extra = actual_dates - expected_dates

print(f"Missing days: {len(missing)}")
if missing:
    print(f"Missing: {sorted(missing)[:10]}")  # Show first 10
    
print(f"\nExtra days: {len(extra)}")
if extra:
    print(f"Extra: {sorted(extra)[:10]}")  # Show first 10

# Check specific known holidays
print("\nKnown holiday check:")
print(f"July 4, 2023 in data? {'2023-07-04' in actual_dates}")  # Should be False
print(f"Dec 25, 2023 in data? {'2023-12-25' in actual_dates}")  # Should be False

Missing days: 1
Missing: ['2023-04-07']

Extra days: 2
Extra: ['2023-10-09', '2023-11-10']

Known holiday check:
July 4, 2023 in data? False
Dec 25, 2023 in data? False


Great question! Let me clarify **exactly** what went wrong and what you should do.

## **What Broke Your Data**

The problem was the **two-step timezone conversion**:

```python
# Step 1: Localize to UTC (assigns midnight UTC)
data.index = data.index.tz_localize('UTC')  
# 2023-01-03 → 2023-01-03 00:00:00+00:00 (midnight UTC)

# Step 2: Convert to US/Eastern
data.index = data.index.tz_convert('US/Eastern')
# 2023-01-03 00:00:00+00:00 → 2023-01-02 19:00:00-05:00 (7PM PREVIOUS DAY!)
```

**The date shifted backwards by one day** because midnight UTC = 7PM Eastern (previous day).

---

## **What You SHOULD Do (Two Options)**

### **Option 1: Skip Timezone Conversion Entirely (Recommended for This Unit)**

```python
# Just use the data as-is (timezone-naive)
data = yf.download(['SPY', 'AAPL', 'MSFT'], start='2019-01-01', end='2024-12-31', auto_adjust=True)['Close']

# No tz_localize, no tz_convert — dates stay correct
```

**Why this works:**
- Stock data is recorded by **date**, not time (market close is the daily price)
- For date-based analysis, timezone doesn't matter
- Your deliverable (`get_last_trading_day()`) only needs the **date**, not the time

---

### **Option 2: Localize DIRECTLY to Market Timezone (If You Need Timezone-Aware)**

```python
# Localize directly to US/Eastern (don't go through UTC)
data.index = data.index.tz_localize('US/Eastern')
# 2023-01-03 → 2023-01-03 00:00:00-05:00 (midnight Eastern, SAME DAY)
```

**Why this works:**
- No round-trip conversion = no date shifting
- Midnight Eastern stays on the same date

---

## **For This Unit: Use Option 1**

**My recommendation:** Skip timezone conversion for Task 3. Here's why:

1. **The learning objective is met:** You understand the *concept* of timezone-naive vs timezone-aware (you answered the self-check correctly)
2. **The hands-on caused issues:** The two-step conversion broke your data
3. **The deliverable doesn't need it:** `get_last_trading_day()` returns a date, not a timestamp with timezone

---

## **Revised Task 3 Approach**

**Keep Task 3 conceptual:**

```markdown
### Task 3: Timezone Concepts (Understanding Check)

**Key takeaways:**
- ✅ Timezone-naive = no timezone info attached
- ✅ Timezone-aware = includes timezone (e.g., UTC, US/Eastern)
- ✅ Cannot mix naive and aware in operations
- ✅ `.tz_localize()` assigns timezone, `.tz_convert()` changes timezone

**For this dataset:**
- Data is timezone-naive (dates only, no times)
- For date-based analysis, this is perfectly fine
- If you needed timezone-aware data, localize directly to US/Eastern (not via UTC)

**Skipping hands-on conversion** to avoid date-shifting issues.
```

---

## **Bottom Line**

**Don't convert for this unit.** The conversion broke your data because of the UTC round-trip. Use timezone-naive data, understand the concepts theoretically, and move on to building `get_last_trading_day()` with clean data.

Does that clear it up? 🎯

**EXACTLY!** You've nailed it. Let me confirm your understanding with precision:

## **What Happened (The Date-Shifting Chain)**

1. **Original data:** `2023-01-03` (just a date, or implicitly midnight with no timezone)

2. **After `tz_localize('UTC')`:**  
   `2023-01-03 00:00:00+00:00` (midnight UTC)

3. **After `tz_convert('US/Eastern')`:**  
   `2023-01-02 19:00:00-05:00` (7PM on **Jan 2nd** — the previous day!)

4. **When you extract `.date`:**  
   You get `2023-01-02` instead of `2023-01-03` ❌

---

## **Why This Happened**

US/Eastern is **UTC-5** (or UTC-4 during daylight saving).  

**Midnight UTC = 7PM Eastern (previous day)**

So the round-trip conversion shifted every date backwards by one day.

---

## **Why Direct Localization Would Work**

If you had done:

```python
data.index = data.index.tz_localize('US/Eastern')  # Skip UTC entirely
```

You'd get:  
`2023-01-03 00:00:00-05:00` (midnight Eastern on **Jan 3rd** — same day!) ✅

---

**You've understood the core issue perfectly:** The UTC conversion shifted the times, which changed the dates. This is why for **date-only analysis** (like this unit), timezone-naive data is safer — no shifting happens.

Now you're ready to build `get_last_trading_day()` with clean data! 🚀

Want to start on the deliverable function, or do you have any other questions about the timezone mechanics?